The toy dataset contains the following variables:

unit_id: A unique identifier for each unit (treated and control units).
time: The day of observation, ranging from 1 to 28.
treatment: A binary indicator, with 1 representing the treated unit and 0 representing control units.
popularity_term: The popularity of the term associated with the product.
popularity_injected: The popularity of the injected product.
conversion_rate_term: The conversion rate of the term associated with the product.
conversion_rate_injected: The conversion rate of the injected product.
profit: The profit generated at each time step.

In [3]:
#https://medium.com/@nasdag/using-synthetic-control-methods-for-causal-inference-a-step-by-step-guide-7bbcb4f49e2f
import numpy as np
import pandas as pd

np.random.seed(42)

n_units = 6
n_days = 28
n_pre = 14

data = []

for unit in range(n_units):
    for day in range(1, n_days + 1):
        treatment = 1 if unit == 0 else 0
        time_trend = day / n_days

        popularity_term = np.random.normal(loc=50 + 10 * time_trend, scale=5)
        popularity_injected = np.random.normal(loc=30 + 10 * time_trend, scale=3)
        conversion_rate_term = np.random.normal(loc=0.1 + 0.02 * time_trend, scale=0.01)
        conversion_rate_injected = np.random.normal(loc=0.05 + 0.01 * time_trend, scale=0.005)

        if unit == 0 and day > n_pre:
            conversion_rate_injected += 0.03

        profit = (popularity_term * conversion_rate_term + popularity_injected * conversion_rate_injected) * 10

        data.append([unit, day, treatment, popularity_term, popularity_injected,
                     conversion_rate_term, conversion_rate_injected, profit])

columns = ['unit_id', 'time', 'treatment', 'popularity_term', 'popularity_injected',
           'conversion_rate_term', 'conversion_rate_injected', 'profit']

df = pd.DataFrame(data, columns=columns)
df

,unit_id,time,treatment,popularity_term,popularity_injected,conversion_rate_term,conversion_rate_injected,profit
0,0,1,1,52.840714,29.942350,0.107191,0.057972,73.998846
1,0,2,1,49.543519,30.011875,0.117221,0.054551,74.447175
2,0,3,1,48.724057,32.699109,0.097509,0.048743,63.448639
3,0,4,1,52.638383,25.688731,0.085608,0.048617,57.551773
4,0,5,1,46.721559,32.728456,0.094491,0.044724,58.785295
...,...,...,...,...,...,...,...,...
163,5,24,0,57.226985,35.251851,0.142876,0.058868,102.515778
164,5,25,0,58.998218,38.856196,0.119838,0.058207,93.319215
165,5,26,0,56.417404,37.645137,0.118244,0.056569,88.005460
166,5,27,0,56.078628,39.962148,0.116736,0.067163,92.303621


In [5]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Convert time to day of the week (0-6)
df['day_of_week'] = df['time'] % 7

# Apply one-hot encoding to the day_of_week
encoder = OneHotEncoder(sparse=False)
encoded_days = encoder.fit_transform(df[['day_of_week']])
day_columns = [f'day_{i}' for i in range(encoded_days.shape[1])]
encoded_days_df = pd.DataFrame(encoded_days, columns=day_columns)

# Combine the original DataFrame with the one-hot encoded day of the week
df_with_days = pd.concat([df, encoded_days_df], axis=1)

# Standardize the time series for each variable and each unit independently
scaler = StandardScaler()
scaled_df = df_with_days.copy()

for unit in scaled_df['unit_id'].unique():
    unit_data = scaled_df[scaled_df['unit_id'] == unit]
    for column in ['popularity_term', 'popularity_injected', 'conversion_rate_term', 'conversion_rate_injected', 'profit'] + day_columns:
        scaled_df.loc[unit_data.index, column] = scaler.fit_transform(unit_data[[column]])

# Calculate the mean and standard deviation of the treated unit's profit during the pre-period
treated_pre_period = df_with_days[(df_with_days['treatment'] == 1) & (df_with_days['time'] <= n_pre)]
mean_treated_profit_pre = treated_pre_period['profit'].mean()
std_treated_profit_pre = treated_pre_period['profit'].std()

# Define a function to rescale the standardized time series
def rescale_time_series(series, mean, std):
    return (series * std) + mean

# Rescale all standardized variables for all units
rescaled_df = scaled_df.copy()
for column in ['popularity_term', 'popularity_injected', 'conversion_rate_term', 'conversion_rate_injected', 'profit'] + day_columns:
    rescaled_df[column] = rescale_time_series(scaled_df[column], mean_treated_profit_pre, std_treated_profit_pre)

In [6]:
rescaled_df

,unit_id,time,treatment,popularity_term,popularity_injected,conversion_rate_term,conversion_rate_injected,profit,day_of_week,day_0,day_1,day_2,day_3,day_4,day_5,day_6
0,0,1,1,67.853558,61.758115,69.520676,65.279398,65.551664,1,67.481850,90.426278,67.481850,67.481850,67.481850,67.481850,67.481850
1,0,2,1,61.294549,61.890831,76.088306,63.755008,65.776979,2,67.481850,67.481850,90.426278,67.481850,67.481850,67.481850,67.481850
2,0,3,1,59.664418,67.020512,63.180296,61.166549,60.249480,3,67.481850,67.481850,67.481850,90.426278,67.481850,67.481850,67.481850
3,0,4,1,67.451067,53.638350,55.387357,61.110558,57.285910,4,67.481850,67.481850,67.481850,67.481850,90.426278,67.481850,67.481850
4,0,5,1,55.680910,67.076533,61.204353,59.375790,57.905838,5,67.481850,67.481850,67.481850,67.481850,67.481850,90.426278,67.481850
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,5,24,0,75.122481,69.913674,88.208624,75.744023,87.097996,3,67.481850,67.481850,67.481850,90.426278,67.481850,67.481850,67.481850
164,5,25,0,78.345012,77.248006,75.535361,74.905454,80.261974,4,67.481850,67.481850,67.481850,67.481850,90.426278,67.481850,67.481850
165,5,26,0,73.649553,74.783673,74.658463,72.826417,76.312135,5,67.481850,67.481850,67.481850,67.481850,67.481850,90.426278,67.481850
166,5,27,0,73.033193,79.498461,73.828950,86.271710,79.507060,6,67.481850,67.481850,67.481850,67.481850,67.481850,67.481850,90.426278


In [7]:
from scipy.spatial.distance import cdist

# Filter the pre-period data for control units
control_pre_period = rescaled_df[(rescaled_df['treatment'] == 0) & (rescaled_df['time'] <= n_pre)]

# Calculate the daily Euclidean distance between the treated unit and each control unit during the pre-period
daily_distances = []

for unit in control_pre_period['unit_id'].unique():
    unit_data = control_pre_period[control_pre_period['unit_id'] == unit][['popularity_term', 'popularity_injected', 'conversion_rate_term', 'conversion_rate_injected', 'profit']]
    treated_data = treated_pre_period[['popularity_term', 'popularity_injected', 'conversion_rate_term', 'conversion_rate_injected', 'profit']]
    daily_distance = cdist(unit_data, treated_data, metric='euclidean').sum()
    daily_distances.append((unit, daily_distance))

# Sort the control units by their daily Euclidean distances and select the two closest control units
closest_units = sorted(daily_distances, key=lambda x: x[1])[:2]
closest_units_indices = [unit for unit, distance in closest_units]

# Filter the data to include only the treated unit and the two closest control units
selected_units = rescaled_df[rescaled_df['unit_id'].isin([0] + closest_units_indices)]

print("Closest control units:", closest_units_indices)

Closest control units: [2, 1]


In [8]:
selected_units 

,unit_id,time,treatment,popularity_term,popularity_injected,conversion_rate_term,conversion_rate_injected,profit,day_of_week,day_0,day_1,day_2,day_3,day_4,day_5,day_6
0,0,1,1,67.853558,61.758115,69.520676,65.279398,65.551664,1,67.481850,90.426278,67.481850,67.481850,67.481850,67.481850,67.481850
1,0,2,1,61.294549,61.890831,76.088306,63.755008,65.776979,2,67.481850,67.481850,90.426278,67.481850,67.481850,67.481850,67.481850
2,0,3,1,59.664418,67.020512,63.180296,61.166549,60.249480,3,67.481850,67.481850,67.481850,90.426278,67.481850,67.481850,67.481850
3,0,4,1,67.451067,53.638350,55.387357,61.110558,57.285910,4,67.481850,67.481850,67.481850,67.481850,90.426278,67.481850,67.481850
4,0,5,1,55.680910,67.076533,61.204353,59.375790,57.905838,5,67.481850,67.481850,67.481850,67.481850,67.481850,90.426278,67.481850
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,2,24,0,78.772017,77.073276,75.612289,74.794195,79.264070,3,67.481850,67.481850,67.481850,90.426278,67.481850,67.481850,67.481850
80,2,25,0,75.434946,84.397528,69.422908,91.533186,77.476446,4,67.481850,67.481850,67.481850,67.481850,90.426278,67.481850,67.481850
81,2,26,0,68.731381,72.313083,81.257510,82.563027,77.323978,5,67.481850,67.481850,67.481850,67.481850,67.481850,90.426278,67.481850
82,2,27,0,79.788696,86.337016,74.906479,70.760233,80.025087,6,67.481850,67.481850,67.481850,67.481850,67.481850,67.481850,90.426278


In [10]:
import numpy as np
from sklearn.linear_model import LinearRegression
from scipy.spatial.distance import cdist

# Select pre-treatment variables, add one-hot-encoded day of the week columns to the features
features = ['popularity_term', 'popularity_injected', 'conversion_rate_term', 'conversion_rate_injected']
day_columns = [f'day_{i}' for i in range(6)]
features += day_columns

# Filter pre-treatment data for the treated unit and the two closest control units
pre_treatment_data = selected_units[selected_units['time'] <= n_pre]

# Estimate a linear regression model using pre-treatment data
X_pre_treatment = pre_treatment_data[features]
y_pre_treatment = pre_treatment_data['profit']

reg = LinearRegression().fit(X_pre_treatment, y_pre_treatment)

# Calculate distances between each record of a control unit in the post-period and the pre-period test
post_treatment_data = selected_units[selected_units['time'] > n_pre]
distances = []
for _, row in post_treatment_data.iterrows():
    control_unit_data = row[features].values.reshape(1, -1)
    treated_unit_data = treated_pre_period[features].values
    distance = cdist(control_unit_data, treated_unit_data, metric='euclidean').min()
    distances.append(distance)

post_treatment_data['distance'] = distances
post_treatment_data['weight'] = 1 / post_treatment_data['distance']

# Predict the synthetic outcome using the weights based on similarity to the pre-period test
X_post_treatment = post_treatment_data[features]
predicted_outcome = reg.predict(X_post_treatment)
post_treatment_data['predicted_profit'] = predicted_outcome

# Create a synthetic control group using the post-treatment data and the predicted outcomes
synthetic_control_group = post_treatment_data[post_treatment_data['treatment'] == 0]

# Calculate the weighted predicted profit for the synthetic control group
synthetic_control_group['weighted_predicted_profit'] = synthetic_control_group['predicted_profit'] * synthetic_control_group['weight'] / synthetic_control_group['weight'].sum()
synthetic_control_group = synthetic_control_group.groupby('time').agg({'weighted_predicted_profit': 'sum'})

# Compare the observed outcome of the treated unit to the predicted outcome from the synthetic control group
treated_unit_data = post_treatment_data[post_treatment_data['unit_id'] == 0]
treated_unit_observed_outcome = treated_unit_data['profit'].mean()
treated_unit_predicted_outcome = synthetic_control_group['weighted_predicted_profit'].sum()

# Calculate the causal effect
causal_effect = np.mean(treated_unit_observed_outcome - treated_unit_predicted_outcome)
print("Estimated causal effect of the treatment:", causal_effect)
#By implementing this method, we can estimate the causal effect of the treatment and identify the key features that have a significant impact on the outcome. This information can be useful for decision-making, resource allocation, and further research.

# Extract feature coefficients from the linear regression model
feature_coefficients = reg.coef_

# Pair features with their coefficients and sort by magnitude
feature_importances = sorted(zip(features, feature_coefficients), key=lambda x: abs(x[1]), reverse=True)

# Print feature importances
for feature, coefficient in feature_importances:
    print(f"{feature}: {coefficient}")

Estimated causal effect of the treatment: 2.1794018770427073
conversion_rate_term: 0.48615930116996653
popularity_term: 0.36882109123060747
conversion_rate_injected: 0.21389967192606704
popularity_injected: 0.16761603572927591
day_5: 0.05764054482398683
day_3: 0.04927724215132885
day_1: 0.04718961899492364
day_0: 0.04406374384687629
day_4: 0.03180909904619391
day_2: 0.024708488114425657


C:\Users\jagap\AppData\Local\Temp\ipykernel_5836\120034310.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  post_treatment_data['distance'] = distances
C:\Users\jagap\AppData\Local\Temp\ipykernel_5836\120034310.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  post_treatment_data['weight'] = 1 / post_treatment_data['distance']
C:\Users\jagap\AppData\Local\Temp\ipykernel_5836\120034310.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row

In [11]:
# Extract feature coefficients from the linear regression model
feature_coefficients = reg.coef_

# Pair features with their coefficients and sort by magnitude
feature_importances = sorted(zip(features, feature_coefficients), key=lambda x: abs(x[1]), reverse=True)

# Print feature importances
for feature, coefficient in feature_importances:
    print(f"{feature}: {coefficient}")

conversion_rate_term: 0.48615930116996653
popularity_term: 0.36882109123060747
conversion_rate_injected: 0.21389967192606704
popularity_injected: 0.16761603572927591
day_5: 0.05764054482398683
day_3: 0.04927724215132885
day_1: 0.04718961899492364
day_0: 0.04406374384687629
day_4: 0.03180909904619391
day_2: 0.024708488114425657


In [14]:
from scipy.optimize import minimize

def weighted_mse_stacked(w, treated, control):
    synthetic_control = (control * w).sum(axis=1)
    return ((treated - synthetic_control) ** 2).mean()

# Stack all variables for the treated unit and control units
treated_pre_period_stacked = np.hstack([treated_pre_period[var].values # for var in variables])
treated_pre_period_stacked

SyntaxError: invalid syntax (3811276828.py, line 9)

In [12]:
from scipy.optimize import minimize

def weighted_mse_stacked(w, treated, control):
    synthetic_control = (control * w).sum(axis=1)
    return ((treated - synthetic_control) ** 2).mean()

# Stack all variables for the treated unit and control units
treated_pre_period_stacked = np.hstack([treated_pre_period[var].values for var in variables])

control_pre_period_stacked = []
for unit_id in selected_units['unit_id'].unique():
    if unit_id != 0:
        control_unit_data = selected_units[(selected_units['unit_id'] == unit_id) & (selected_units['time'] <= n_pre)]
        stacked_data = np.hstack([control_unit_data[var].values for var in variables])
        control_pre_period_stacked.append(stacked_data)
control_pre_period_stacked = np.column_stack(control_pre_period_stacked)

# Estimate the weights
initial_weights = np.array([0.5, 0.5])
res = minimize(weighted_mse_stacked, initial_weights, args=(treated_pre_period_stacked, control_pre_period_stacked), bounds=[(0, 1), (0, 1)], constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1})

# Apply the weights to the control units
optimal_weights = res.x
synthetic_control_pre_period_stacked = (control_pre_period_stacked * optimal_weights).sum(axis=1)

# Calculate the predicted outcome without the intervention
control_post_period_stacked = []
for unit_id in selected_units['unit_id'].unique():
    if unit_id != 0:
        control_unit_data = selected_units[(selected_units['unit_id'] == unit_id) & (selected_units['time'] > n_pre)]
        stacked_data = np.hstack([control_unit_data[var].values for var in ['profit'])
        control_post_period_stacked.append(stacked_data)
control_post_period_stacked = np.column_stack(control_post_period_stacked)

synthetic_control_post_period_stacked = (control_post_period_stacked * optimal_weights).sum(axis=1)

# Compare the observed outcome after the intervention to the predicted outcome without the intervention
observed_outcome_stacked = np.hstack([test_profit_data[test_profit_data['time'] > n_pre][var].values for var in variables])
predicted_outcome_stacked = synthetic_control_post_period_stacked

# Estimate the causal effect
causal_effect_stacked = np.mean(observed_outcome_stacked - predicted_outcome_stacked)
print("Estimated causal effect of the treatment using stacked variables:", causal_effect_stacked)

SyntaxError: closing parenthesis ')' does not match opening parenthesis '[' (1662386016.py, line 31)